# B6.3 — Baselines I: vanilla SAM and TFOC (re-run by us)

Pre-declared in `results/B6_0_Setup_and_Rules/B6_rule_file.json`.

| Method | What | Datasets |
|---|---|---|
| **Vanilla SAM** (literature anchor) | official `segment-anything`, SAM ViT-B, default automatic mask generator, count = number of masks (TFOC's "SAM" row) | FSC-147 test, CARPK test |
| **TFOC** (Shi et al., WACV 2024) | official code (commit `1da2dbb`), **text prompts**: CLIP-Surgery similarity → largest region → boxes → prior-guided SAM ViT-B masks; count = number of masks | FSC-147 test, CARPK test; VOC 2007 and OmniCount-191: present classes (all images) + all classes on the fixed subsets (500 VOC images; OmniCount Fruits) |

TFOC's text mode exists only in its FSC-147 script; for the other datasets the **same text procedure** is applied unchanged with each dataset's class names (the same prompt names as our pipelines).
Compatibility patches for current packages (behaviour-neutral) are applied and printed in Step 0.2.

**Order:** vanilla SAM (FSC-147, CARPK) → TFOC FSC-147 + **reproduction gate** (stops if MAE is more than 3 from the published 24.79) → TFOC CARPK, VOC, OmniCount.
**Runtime (L4):** ~1 h vanilla SAM + ~15–20 h TFOC. **Resumable** — Run all again to continue. A separate runtime from B6_2 (can run in parallel).

In [1]:
# Step 0.1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 0.2 — Paths, our repo (loaders), TFOC at a fixed commit + documented compatibility patches, weights
# ============================================================
import hashlib, json, shutil, subprocess, sys, tarfile, time, urllib.request, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np, pandas as pd
import torch
assert torch.cuda.is_available(), "Use a GPU runtime (L4)."
GPU_NAME = torch.cuda.get_device_name(0); print("GPU:", GPU_NAME)
PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"; DATASETS = PROJECT_ROOT / "datasets"
OUT_DIR = RESULTS_ROOT / "B6_3_Baselines_SAM_TFOC"; OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/data"); LOCAL.mkdir(parents=True, exist_ok=True)

!pip install -q ftfy regex selective-search "git+https://github.com/facebookresearch/segment-anything.git"

REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
subprocess.run(["rm", "-rf", str(REPO_DIR)])
subprocess.run(["git", "clone", "-q", "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git", str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

TFOC_COMMIT = "1da2dbb"
TFOC_DIR = Path("/content/tfoc")
if not TFOC_DIR.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/shizenglin/training-free-object-counter.git", str(TFOC_DIR)], check=True)
    subprocess.run(["git", "-C", str(TFOC_DIR), "checkout", "-q", TFOC_COMMIT], check=True)
    # --- Compatibility patches (no change to the method) ---
    p = TFOC_DIR / "clip" / "clip.py"
    p.write_text(p.read_text().replace("from pkg_resources import packaging", "import packaging.version  # patched: pkg_resources deprecated"))
    p = TFOC_DIR / "utils.py"
    p.write_text(p.read_text().replace("from scipy.ndimage.filters import gaussian_filter",
                                       "from scipy.ndimage import gaussian_filter  # patched: scipy.ndimage.filters removed"))
print(subprocess.run(["git", "-C", str(TFOC_DIR), "diff"], capture_output=True, text=True).stdout)
TFOC_HEAD = subprocess.check_output(["git", "-C", str(TFOC_DIR), "rev-parse", "HEAD"], text=True).strip()
assert TFOC_HEAD.startswith(TFOC_COMMIT), TFOC_HEAD

# --- SAM ViT-B checkpoint (cached on Drive; file name carries its SHA-256 prefix 01ec64)
SAM_B = PROJECT_ROOT / "Checkpoints" / "SAM" / "sam_vit_b_01ec64.pth"
SAM_B.parent.mkdir(parents=True, exist_ok=True)
if not SAM_B.exists():
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth", SAM_B)
h = hashlib.md5(SAM_B.read_bytes()).hexdigest()
assert h.startswith("01ec64"), h          # SAM checkpoint names carry the MD5 prefix
(TFOC_DIR / "pretrain").mkdir(exist_ok=True)
if not (TFOC_DIR / "pretrain" / "sam_vit_b_01ec64.pth").exists():
    shutil.copy2(SAM_B, TFOC_DIR / "pretrain" / "sam_vit_b_01ec64.pth")

# --- CLIP ViT-B/16 weights for CLIP-Surgery (verified by CLIP's own SHA-256 check); current host as fallback
CLIP_SHA = "5806e77cd80f8b59890b7e101eabd078d9fb84e6937f9e85e4ecb61988df416f"
CLIP_CACHE = Path.home() / ".cache" / "clip"; CLIP_CACHE.mkdir(parents=True, exist_ok=True)
CLIP_FILE = CLIP_CACHE / "ViT-B-16.pt"; CLIP_DRIVE = PROJECT_ROOT / "Checkpoints" / "CLIP" / "ViT-B-16.pt"
if CLIP_DRIVE.exists() and not CLIP_FILE.exists():
    shutil.copy2(CLIP_DRIVE, CLIP_FILE)
if not CLIP_FILE.exists() or hashlib.sha256(CLIP_FILE.read_bytes()).hexdigest() != CLIP_SHA:
    for host in ["openaipublic.azureedge.net", "openaipublic.blob.core.windows.net"]:
        try:
            urllib.request.urlretrieve(f"https://{host}/clip/models/{CLIP_SHA}/ViT-B-16.pt", CLIP_FILE)
            if hashlib.sha256(CLIP_FILE.read_bytes()).hexdigest() == CLIP_SHA:
                print("CLIP weights from", host); break
        except Exception as e:
            print("CLIP download failed from", host, "-", type(e).__name__)
assert hashlib.sha256(CLIP_FILE.read_bytes()).hexdigest() == CLIP_SHA, "CLIP weights not available"
CLIP_DRIVE.parent.mkdir(parents=True, exist_ok=True)
if not CLIP_DRIVE.exists():
    shutil.copy2(CLIP_FILE, CLIP_DRIVE)
print("Repo:", REPO_COMMIT[:10], "| TFOC:", TFOC_HEAD[:7], "| SAM ViT-B and CLIP ViT-B/16 verified")

GPU: NVIDIA L4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
diff --git a/clip/clip.py b/clip/clip.py
index 201964d..2a07246 100644
--- a/clip/clip.py
+++ b/clip/clip.py
@@ -3,7 +3,7 @@ import os
 import urllib
 import warnings
 from typing import Union, List
-from pkg_resources import packaging
+import packaging.version  # patched: pkg_resources deprecated
 
 import torch
 from PIL import Image
diff --git a/utils.py b/utils.py
index 0c40365..cc67a97 100644
--- a/utils.py
+++ b/utils.py
@@ -10,7 +10,7 @@ import matplotlib.pyplot as plt
 import matplotlib.patches as patches
 matplotlib.use('agg')
 import scipy.spatial as T
-from scipy.ndimage.filters import gaussian_filter
+from scipy.ndimage import gaussian_filter  # patched: scipy.ndimage.filters removed
 import clip
 from skimage.measure import label, find_contours
 from selective_search import selective_search

CLIP weights from openaipublic.azu

In [3]:
# ============================================================
# Step 0.3 — Load models: vanilla SAM (official package) and TFOC (its own SAM fork + CLIP-Surgery)
# ============================================================
import cv2
from tqdm.auto import tqdm
from segment_anything import sam_model_registry as sam_registry_official, SamAutomaticMaskGenerator as SamAMG_official
sam_v = sam_registry_official["vit_b"](checkpoint=str(SAM_B)).to("cuda")
vanilla_amg = SamAMG_official(sam_v)                                       # default parameters (TFOC's "SAM" row)

sys.path.insert(0, str(TFOC_DIR))
import clip                                                                 # TFOC's bundled CLIP-Surgery
from shi_segment_anything import sam_model_registry as sam_registry_tfoc
from shi_segment_anything.automatic_mask_generator_text import SamAutomaticMaskGenerator as TFOCTextAMG
import utils as tfoc_utils
assert Path(clip.__file__).resolve().is_relative_to(TFOC_DIR), clip.__file__
clip_model, _ = clip.load("CS-ViT-B/16", device="cuda"); clip_model.eval()
sam_t = sam_registry_tfoc["vit_b"](checkpoint=str(TFOC_DIR / "pretrain" / "sam_vit_b_01ec64.pth")).to("cuda")
tfoc_amg = TFOCTextAMG(model=sam_t)

def read_rgb(path):
    return cv2.cvtColor(cv2.imread(str(path)), cv2.COLOR_BGR2RGB)        # as in TFOC's scripts

def count_vanilla_sam(img):
    return len(vanilla_amg.generate(img))

def count_tfoc_text(img, text):
    """TFOC text mode exactly as in main-fsc147.py: CLIP-Surgery boxes -> prior-guided masks; count = masks."""
    boxes = tfoc_utils.get_clip_bboxs(clip_model, img, text, "cuda")
    return len(tfoc_amg.generate(img, boxes))
print("Models loaded: vanilla SAM ViT-B; TFOC (CLIP-Surgery CS-ViT-B/16 + SAM ViT-B fork)")

Models loaded: vanilla SAM ViT-B; TFOC (CLIP-Surgery CS-ViT-B/16 + SAM ViT-B fork)


In [4]:
# ============================================================
# Step 0.4 — Test items and the queries per image (rule file: present classes everywhere; all classes on the subsets)
# ============================================================
sys.path.insert(0, str(REPO_DIR))
import src.datasets as D
RULE = json.load(open(RESULTS_ROOT / "B6_0_Setup_and_Rules" / "B6_rule_file.json"))
SUBSETS = json.load(open(RESULTS_ROOT / "B6_0_Setup_and_Rules" / "B6_0_baseline_subsets.json"))
VOC_SUB, FRUITS_SUB = set(SUBSETS["voc_standard_500"]), set(SUBSETS["omnicount_fruits_test"])
ITEMS = {"fsc147": [], "carpk": [], "voc": [], "omnicount": []}

fsc = D.FSC147Dataset(DATASETS / "FSC147" / "raw", split="test", load_images=False)
for i in range(len(fsc)):
    s = fsc[i]; c = next(iter(s["categories"]))
    ITEMS["fsc147"].append({"key": f"{i:05d}", "image_path": s["image_path"], "query": [c], "prompts": {c: c},
                            "meta": {"dataset": "fsc147", "image_id": s["image_id"], "gt": {c: s["categories"][c]["count"]}}})
CARPK_LOCAL = LOCAL / "carpk"
if not any(CARPK_LOCAL.rglob("test_annotations.json")):
    with zipfile.ZipFile(DATASETS / "CARPK_kaggle_coco.zip") as z:
        z.extractall(CARPK_LOCAL)
cds = D.CARPKDataset(CARPK_LOCAL, "test", load_images=False)
for i in range(len(cds)):
    s = cds[i]
    ITEMS["carpk"].append({"key": Path(s["image_path"]).stem, "image_path": s["image_path"], "query": ["car"], "prompts": {"car": "car"},
                           "meta": {"dataset": "carpk", "image_id": s["image_id"], "gt": {"car": s["categories"]["car"]["count"]}}})
VOC_LOCAL = LOCAL / "voc2007"
if not (VOC_LOCAL / "VOCdevkit" / "VOC2007" / "ImageSets").exists():
    with tarfile.open(DATASETS / "VOCtest_06-Nov-2007.tar") as t:
        t.extractall(VOC_LOCAL, filter="data")
voc = D.VOC2007CountingDataset(VOC_LOCAL, "test", include_difficult=True, load_images=False)
for i in range(len(voc)):
    s = voc[i]; vid = Path(s["image_path"]).stem
    present = list(s["categories"])                                         # >= 1 annotated object (primary convention)
    query = list(D.VOC_CLASSES) if vid in VOC_SUB else present
    ITEMS["voc"].append({"key": vid, "image_path": s["image_path"], "query": query, "prompts": {c: D.voc_prompt_name(c) for c in query},
                         "meta": {"dataset": "voc2007", "domain": "VOC2007", "image_id": s["image_id"], "in_standard_subset": vid in VOC_SUB,
                                  "gt_all": {c: v["count_all"] for c, v in s["categories"].items()},
                                  "gt_nondifficult": {c: v["count_nondifficult"] for c, v in s["categories"].items() if v["count_nondifficult"] > 0}}})
om = D.OmniCount191Dataset(DATASETS / "OmniCount-191", split="test", load_images=False)
for i in range(len(om)):
    s = om[i]; d = s["domain"]; present = list(s["categories"])
    query = list(om.domain_vocabulary[d]) if s["image_id"] in FRUITS_SUB else present
    ITEMS["omnicount"].append({"key": s["image_id"].replace("/", "__"), "image_path": s["image_path"], "query": query,
                               "prompts": {c: D.omnicount_prompt_name(c) for c in query},
                               "meta": {"dataset": "omnicount", "domain": d, "image_id": s["image_id"], "in_standard_subset": s["image_id"] in FRUITS_SUB,
                                        "vocabulary": list(om.domain_vocabulary[d]), "gt": {c: v["count"] for c, v in s["categories"].items()}}})
inv = RULE["inventory"]
assert len(ITEMS["fsc147"]) == 1190 and len(ITEMS["carpk"]) == 459 and len(ITEMS["voc"]) == 4952 and len(ITEMS["omnicount"]) == 1957
assert sum(it["meta"]["in_standard_subset"] for it in ITEMS["voc"]) == 500 and sum(it["meta"]["in_standard_subset"] for it in ITEMS["omnicount"]) == len(FRUITS_SUB)
print({k: f"{len(v)} images, {sum(len(it['query']) for it in v)} queries" for k, v in ITEMS.items()})

{'fsc147': '1190 images, 1190 queries', 'carpk': '459 images, 459 queries', 'voc': '4952 images, 16868 queries', 'omnicount': '1957 images, 4952 queries'}


In [5]:
# ============================================================
# Step 0.5 — Resumable runner (one JSON per image, common format)
# ============================================================
def run_method(method, items, rows_dir, fn):
    rows_dir = Path(rows_dir); rows_dir.mkdir(parents=True, exist_ok=True)
    todo = [it for it in items if not (rows_dir / f"{it['key']}.json").exists()]
    for it in tqdm(todo, desc=f"{method} {rows_dir.parent.name}"):
        t0 = time.time(); img = read_rgb(it["image_path"]); counts, errors = {}, {}
        for c in it["query"]:
            try:
                counts[c] = int(fn(img, it["prompts"][c]))
            except Exception as e:                                          # recorded, counted as 0 in B6_5
                counts[c] = 0; errors[c] = f"{type(e).__name__}: {str(e)[:200]}"
        rec = {"key": it["key"], "image_path": str(it["image_path"]), "classes": it["query"], "meta": it["meta"],
               "version": method, "counts": {method: counts}, "diag": {"t_total_s": time.time() - t0, "errors": errors}}
        tmp = rows_dir / f"{it['key']}.json.tmp"; tmp.write_text(json.dumps(rec)); tmp.replace(rows_dir / f"{it['key']}.json")
    return len(todo)

def single_metrics(rows_dir, method):
    rows = [json.load(open(p)) for p in Path(rows_dir).glob("*.json")]
    gt = np.array([r["meta"]["gt"][r["classes"][0]] for r in rows], float)
    pr = np.array([r["counts"][method][r["classes"][0]] for r in rows], float); e = pr - gt
    return {"images": len(rows), "MAE": np.abs(e).mean(), "RMSE": np.sqrt((e ** 2).mean()), "NAE": (np.abs(e) / gt).mean(),
            "SRE": np.sqrt(((e ** 2) / gt).mean()), "errors": sum(len(r["diag"]["errors"]) for r in rows)}

In [6]:
# ============================================================
# Step 1.1 — Vanilla SAM on FSC-147 test and CARPK test (literature anchor; published 42.48 / 16.97 MAE)
# ============================================================
for name in ["fsc147", "carpk"]:
    run_method("SAM", ITEMS[name], OUT_DIR / name / "rows_sam", lambda img, text: count_vanilla_sam(img))
PUB_SAM = {"fsc147": {"MAE": 42.48, "RMSE": 137.50, "NAE": 1.14, "SRE": 8.13}, "carpk": {"MAE": 16.97, "RMSE": 20.57, "NAE": 0.70, "SRE": 5.30}}
for name in ["fsc147", "carpk"]:
    m = single_metrics(OUT_DIR / name / "rows_sam", "SAM")
    print(f"Vanilla SAM {name}: ours MAE {m['MAE']:.2f} RMSE {m['RMSE']:.2f} NAE {m['NAE']:.2f} SRE {m['SRE']:.2f} | "
          f"published (TFOC Table 1) {PUB_SAM[name]} | errors {m['errors']}")

SAM fsc147:   0%|          | 0/1190 [00:00<?, ?it/s]

SAM carpk:   0%|          | 0/459 [00:00<?, ?it/s]

Vanilla SAM fsc147: ours MAE 46.72 RMSE 138.70 NAE 1.28 SRE 8.81 | published (TFOC Table 1) {'MAE': 42.48, 'RMSE': 137.5, 'NAE': 1.14, 'SRE': 8.13} | errors 0
Vanilla SAM carpk: ours MAE 17.27 RMSE 20.94 NAE 0.72 SRE 5.48 | published (TFOC Table 1) {'MAE': 16.97, 'RMSE': 20.57, 'NAE': 0.7, 'SRE': 5.3} | errors 0


In [7]:
# ============================================================
# Step 2.1 — TFOC text prompts on FSC-147 test + REPRODUCTION GATE (published MAE 24.79, RMSE 137.15)
# ============================================================
GATE_TOLERANCE = 3.0          # stop if |MAE - 24.79| > 3 (set OVERRIDE_GATE = True only after reviewing why)
OVERRIDE_GATE = False
run_method("TFOC", ITEMS["fsc147"], OUT_DIR / "fsc147" / "rows_tfoc", count_tfoc_text)
m = single_metrics(OUT_DIR / "fsc147" / "rows_tfoc", "TFOC")
print(f"TFOC FSC-147 test (text): MAE {m['MAE']:.2f} RMSE {m['RMSE']:.2f} NAE {m['NAE']:.2f} SRE {m['SRE']:.2f} | "
      f"published 24.79 / 137.15 / 0.37 / 4.52 | errors {m['errors']}")
REPRO = {"tfoc_fsc147": m, "published": {"MAE": 24.79, "RMSE": 137.15, "NAE": 0.37, "SRE": 4.52}}
json.dump(REPRO, open(OUT_DIR / "B6_3_tfoc_reproduction.json", "w"), indent=2)
if abs(m["MAE"] - 24.79) > GATE_TOLERANCE and not OVERRIDE_GATE:
    raise SystemExit(f"REPRODUCTION GATE: MAE {m['MAE']:.2f} is more than {GATE_TOLERANCE} from 24.79 — stop and review before the other datasets.")
print("REPRODUCTION GATE: PASS")

TFOC fsc147:   0%|          | 0/1190 [00:00<?, ?it/s]

/content/tfoc/utils.py:212: RuntimeWarning: invalid value encountered in scalar divide
  score = np.sum(img_gray[Ymin:Ymax,Xmin:Xmax])/((Xmax-Xmin)*(Ymax-Ymin))
/content/tfoc/utils.py:227: RuntimeWarning: invalid value encountered in scalar divide
  score = np.sum(img_gray[Ymin:Ymax,Xmin:Xmax])/((Xmax-Xmin)*(Ymax-Ymin))


TFOC FSC-147 test (text): MAE 24.70 RMSE 137.28 NAE 0.35 SRE 4.43 | published 24.79 / 137.15 / 0.37 / 4.52 | errors 0
REPRODUCTION GATE: PASS


In [8]:
# ============================================================
# Step 3.1 — TFOC text prompts on CARPK, VOC 2007 and OmniCount-191 (resumable)
# ============================================================
RUN = ["carpk", "voc", "omnicount"]
for name in RUN:
    t0 = time.time()
    n = run_method("TFOC", ITEMS[name], OUT_DIR / name / "rows_tfoc", count_tfoc_text)
    print(f"{name}: processed now {n} | {(time.time() - t0) / 3600:.2f} h")

TFOC carpk:   0%|          | 0/459 [00:00<?, ?it/s]

carpk: processed now 459 | 0.86 h


TFOC voc:   0%|          | 0/4952 [00:00<?, ?it/s]

/content/tfoc/shi_segment_anything/automatic_mask_generator_text.py:288: RuntimeWarning: divide by zero encountered in scalar floor_divide
  points_per_side = (self.points_per_side//(1.2*mask_size)+1)*self.points_per_side


voc: processed now 4952 | 9.27 h


TFOC omnicount:   0%|          | 0/1957 [00:00<?, ?it/s]

/content/tfoc/utils.py:131: RuntimeWarning: invalid value encountered in scalar divide
  score = np.sum(similarity_mask[Ymin:Ymax,Xmin:Xmax])/((Xmax-Xmin)*(Ymax-Ymin))


omnicount: processed now 1957 | 3.39 h


In [9]:
# ============================================================
# Step 4.1 — Completeness, progress summary (descriptive; B6_5 runs the pre-declared analysis), metadata
# ============================================================
import src.evaluation as ev
status, summary = {}, []
for name, method in [("fsc147", "SAM"), ("carpk", "SAM"), ("fsc147", "TFOC"), ("carpk", "TFOC"), ("voc", "TFOC"), ("omnicount", "TFOC")]:
    d = OUT_DIR / name / f"rows_{method.lower()}"
    done = len(list(d.glob("*.json"))) if d.exists() else 0
    status[f"{method}|{name}"] = f"{done}/{len(ITEMS[name])}"
    if done < len(ITEMS[name]):
        continue
    if name in ("fsc147", "carpk"):
        summary.append({"dataset": name, "method": method, "protocol": "single-class", **single_metrics(d, method)})
        continue
    rows = [json.load(open(p)) for p in d.glob("*.json")]
    gk = "gt_all" if name == "voc" else "gt"
    present = [{"image_id": r["meta"]["image_id"], "domain": r["meta"]["domain"],
                "categories": {c: {"count": n} for c, n in r["meta"][gk].items()},
                "pred_counts": r["counts"][method], "query_classes": list(r["meta"][gk])} for r in rows]
    sub = [{"image_id": r["meta"]["image_id"], "domain": r["meta"]["domain"],
            "categories": {c: {"count": n} for c, n in r["meta"][gk].items()},
            "pred_counts": r["counts"][method], "query_classes": r["classes"]} for r in rows if r["meta"]["in_standard_subset"]]
    for proto, res in [("present classes (full test)", present), ("standard (subset)", sub)]:
        _, _, m = ev.evaluate_multi_label_counts(ev.build_multi_label_pairs(res))
        summary.append({"dataset": name, "method": method, "protocol": proto, "images": len(res), "MAE_nz": m["MAE_nz"],
                        "mRMSE-nz": m["mRMSE_nz"], "standard mRMSE": m["mRMSE"] if proto.startswith("standard") else np.nan,
                        "FP units": m["false_positive_units"], "errors": sum(len(r["diag"]["errors"]) for r in rows)})
print("Status:", status)
if summary:
    sm = pd.DataFrame(summary); display(sm.round(3)); sm.to_csv(OUT_DIR / "B6_3_progress_summary.csv", index=False)
complete = all(v.split("/")[0] == v.split("/")[1] for v in status.values())
json.dump({"notebook": "B6_3_Baselines_SAM_TFOC", "tfoc_commit": TFOC_HEAD, "repo_commit": REPO_COMMIT, "gpu": GPU_NAME,
           "patches": ["clip/clip.py: pkg_resources.packaging -> packaging.version", "utils.py: scipy.ndimage.filters -> scipy.ndimage"],
           "tfoc_text_templates": ["a photo of many {}.", "a photo of the small {}"], "status": status, "complete": complete,
           "rule_file_recorded_utc": RULE["recorded_utc"], "torch": torch.__version__,
           "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}, open(OUT_DIR / "B6_3_metadata.json", "w"), indent=2)
print("COMPLETE" if complete else "NOT COMPLETE — Run all again to continue", "| saved to", OUT_DIR)

Status: {'SAM|fsc147': '1190/1190', 'SAM|carpk': '459/459', 'TFOC|fsc147': '1190/1190', 'TFOC|carpk': '459/459', 'TFOC|voc': '4952/4952', 'TFOC|omnicount': '1957/1957'}


,dataset,method,protocol,images,MAE,RMSE,NAE,SRE,errors,MAE_nz,mRMSE-nz,standard mRMSE,FP units
0,fsc147,SAM,single-class,1190,46.724,138.697,1.283,8.812,0,NaN,NaN,NaN,NaN
1,carpk,SAM,single-class,459,17.266,20.944,0.721,5.476,0,NaN,NaN,NaN,NaN
2,fsc147,TFOC,single-class,1190,24.698,137.281,0.352,4.430,0,NaN,NaN,NaN,NaN
3,carpk,TFOC,single-class,459,37.488,44.370,0.397,4.163,0,NaN,NaN,NaN,NaN
4,voc,TFOC,present classes (full test),4952,NaN,NaN,NaN,NaN,1,8.647,15.134,NaN,0.0
5,voc,TFOC,standard (subset),500,NaN,NaN,NaN,NaN,1,7.394,12.796,12.678,61919.0
6,omnicount,TFOC,present classes (full test),1957,NaN,NaN,NaN,NaN,0,10.620,15.650,NaN,0.0
7,omnicount,TFOC,standard (subset),303,NaN,NaN,NaN,NaN,0,3.824,4.100,4.567,5700.0


COMPLETE | saved to /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B6_3_Baselines_SAM_TFOC
